# 07: Pointcloud Labeling from Frame Segmentation

This notebook demonstrates how to transfer 2D segmentation results to a 3D pointcloud, using data from Project Aria. The goal is to assign semantic labels to each 3D point by projecting it onto the segmented image and reading the corresponding label.

### In this notebook, we will:
1. Load MPS data: pointcloud and camera trajectory.
2. Synchronize each segmented frame with the closest camera pose.
3. For each 3D point:
    - Transform from world to camera coordinates.
    - Project onto the image plane.
    - Assign a label from the segmentation mask at the projected pixel.
4. Visualize the labeled pointcloud.

As a sample, we will use `kettle_and_forklift_recording.vrs` located in the local `data/raw/kettle_and_forklift/` directory and its corresponding segmentation masks located in `data/outputs/segmentation/kettle_and_forklift/masks` .

### Note:
This approach will also label points behind the object ("tunnel effect"), but is a robust first step for 2D/3D semantic bridging.

## 7.1 Load MPS Pointcloud and Camera Trajectory


In this section, we load the 3D pointcloud and the camera trajectory (poses) produced by MPS.

- The pointcloud is typically stored as `semidense_points.csv` or `semidense_points.csv.gz` and contains $(X, Y, Z)$ coordinates for each point.
- The trajectory is stored as `closed_loop_trajectory.csv` and contains timestamped camera poses (position + orientation).

In [7]:
import os
import pandas as pd
from aria_pylib import find_points_file, find_trajectory_file

# Define the directory containing MPS outputs
MPS_DIR = os.path.join('..', 'data', 'raw', 'kettle_and_forklift', 'mps_kettle_and_forklift_recording_vrs', 'slam')

# Load the pointcloud file
points_path = find_points_file(MPS_DIR)
print(f"Loading pointcloud from: {points_path}")
points_df = pd.read_csv(points_path)

# Load trajectory file
trajectory_path = find_trajectory_file(MPS_DIR)
print(f"Loading trajectory from: {trajectory_path}")
trajectory_df = pd.read_csv(trajectory_path)
print(f"Loaded {len(trajectory_df)} poses.")


Loading pointcloud from: ..\data\raw\kettle_and_forklift\mps_kettle_and_forklift_recording_vrs\slam\semidense_points.csv.gz
Loading trajectory from: ..\data\raw\kettle_and_forklift\mps_kettle_and_forklift_recording_vrs\slam\closed_loop_trajectory.csv
Loaded 50626 poses.


## 7.2 Synchronize trajectory and pointcloud timestamps
Now that we have loaded both the pointcloud and the trajectory, we need to synchronize their timestamps. This step ensures that each 3D point can be associated with the correct camera pose for projection into the image frame.

In [ ]:
# For each point it finds the closest pose in time from the trajectory
# The pointcloud and trajectory share the 'graph_uid' column, which can be used to join them.
trajectory_unique = trajectory_df.drop_duplicates('graph_uid')
points_synced = pd.merge(points_df, trajectory_unique, on='graph_uid', how='left', suffixes=('', '_pose'))
print(f"Synchronized {len(points_synced)} points with unique 'graph_uid' poses.")

Synchronized 239524 points with unique 'graph_uid' poses.


## 7.3 Project 3D points to the image plane
Now we will project each 3D point (in world coordinates) into the corresponding camera image, using the associated pose and camera intrinsics. This allows us to find the pixel location for each point and later assign a segmentation label.